# 03. Damage Segmentation — full CarDD baseline on MPS

Ноутбук обучает YOLOv8-seg на полном baseline-датасете `ml/data/cardd_yolo_seg_full` без negatives и без undersampling, и синхронизирует лучший чекпойнт в `ml/damage_seg/weights` по метрике `mAP50(M)`.

In [ ]:

!pip install -q ultralytics pyyaml matplotlib pandas

In [ ]:

import os
import sys
import json
from pathlib import Path

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if any((cand / marker).exists() for marker in ["pyproject.toml", "requirements.txt", ".git"]):
            return cand
    return start.parent if start.parent.exists() else start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.damage_seg.checkpointing import MetricCheckpointSync, metrics_to_dict

In [ ]:

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if any((cand / m).exists() for m in ["pyproject.toml", "requirements.txt", ".git"]):
            return cand
    return start.parent if start.parent.exists() else start

PROJECT_ROOT = find_project_root(Path("."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_YAML = PROJECT_ROOT / "ml" / "data" / "cardd_yolo_seg_full" / "data.yaml"
DAMAGE_SEG_ROOT = PROJECT_ROOT / "ml" / "damage_seg"
CONFIGS_DIR = DAMAGE_SEG_ROOT / "configs"
WEIGHTS_DIR = DAMAGE_SEG_ROOT / "weights"
REPORTS_DIR = DAMAGE_SEG_ROOT / "reports"
RUNS_DIR = DAMAGE_SEG_ROOT / "runs"
for p in [CONFIGS_DIR, WEIGHTS_DIR, REPORTS_DIR, RUNS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

BEST_METRIC_NAME = "metrics/mAP50(M)"
BEST_WEIGHTS_PATH = WEIGHTS_DIR / "best_damage_seg_full_baseline_mps.pt"
BEST_METRIC_PATH = WEIGHTS_DIR / "best_damage_seg_full_baseline_mps_metrics.json"

DEVICE = "cuda" if os.environ.get("CUDA_VISIBLE_DEVICES") else ("mps" if sys.platform == "darwin" else "cpu")
if Path("/usr/bin/nvidia-smi").exists():
    DEVICE = "cuda"

CFG = {
    "model": "yolov8s-seg.pt",
    "imgsz": 1024 if DEVICE in {"cuda", "mps"} else 768,
    "epochs": 80 if DEVICE in {"cuda", "mps"} else 40,
    "batch": 8 if DEVICE == "cuda" else (2 if DEVICE == "mps" else 2),
    "patience": 15,
    "workers": 0 if DEVICE == "mps" else 2,
    "degrees": 5.0,
    "translate": 0.05,
    "scale": 0.20,
    "fliplr": 0.5,
    "mosaic": 0.25,
    "mixup": 0.0,
    "hsv_h": 0.010,
    "hsv_s": 0.35,
    "hsv_v": 0.20,
    "seed": 42,
    "best_metric": BEST_METRIC_NAME,
}
(CONFIGS_DIR / "yolo_seg_full_baseline_mps.json").write_text(json.dumps(CFG, indent=2), encoding="utf-8")
print("DATA_YAML =", DATA_YAML)
print("DEVICE    =", DEVICE)

In [ ]:

assert DATA_YAML.exists(), f"Missing {DATA_YAML}. Run 01_cardd_to_yolo_full_baseline_v2.ipynb first."
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)
print(data_yaml)

In [ ]:

model = YOLO(CFG["model"])
metric_sync = MetricCheckpointSync(
    metric_name=CFG["best_metric"],
    target_path=BEST_WEIGHTS_PATH,
    metadata_path=BEST_METRIC_PATH,
)
model.add_callback("on_fit_epoch_end", metric_sync)
train_res = model.train(
    data=str(DATA_YAML),
    imgsz=CFG["imgsz"],
    epochs=CFG["epochs"],
    batch=CFG["batch"],
    patience=CFG["patience"],
    workers=CFG["workers"],
    project=str(RUNS_DIR),
    name="damage_seg_full_baseline_mps",
    exist_ok=True,
    device=DEVICE,
    seed=CFG["seed"],
    degrees=CFG["degrees"],
    translate=CFG["translate"],
    scale=CFG["scale"],
    fliplr=CFG["fliplr"],
    mosaic=CFG["mosaic"],
    mixup=CFG["mixup"],
    hsv_h=CFG["hsv_h"],
    hsv_s=CFG["hsv_s"],
    hsv_v=CFG["hsv_v"],
    pretrained=True,
    plots=True,
)
print("Best synced metric:", metric_sync.best_value)
print("Best synced weights:", BEST_WEIGHTS_PATH)

In [ ]:

# Валидация на val и test
best_path = BEST_WEIGHTS_PATH if BEST_WEIGHTS_PATH.exists() else Path(model.trainer.best)
print("best weights:", best_path)

best_model = YOLO(str(best_path))
val_metrics = best_model.val(data=str(DATA_YAML), split="val", device=DEVICE, imgsz=CFG["imgsz"], batch=CFG["batch"])
test_metrics = best_model.val(data=str(DATA_YAML), split="test", device=DEVICE, imgsz=CFG["imgsz"], batch=CFG["batch"])

summary = {
    "best_weights": str(best_path),
    "best_metric_name": CFG["best_metric"],
    "best_metric_sync": json.loads(BEST_METRIC_PATH.read_text(encoding="utf-8")) if BEST_METRIC_PATH.exists() else {},
    "val_metrics": metrics_to_dict(val_metrics),
    "test_metrics": metrics_to_dict(test_metrics),
}
(REPORTS_DIR / "damage_seg_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved best weights to:", BEST_WEIGHTS_PATH)

In [ ]:

# Быстрый sanity-check на нескольких test image
test_dir = DATA_YAML.parent / "images" / "test"
test_imgs = sorted([p for p in test_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:4]
if test_imgs:
    preds = best_model.predict([str(p) for p in test_imgs], device=DEVICE, imgsz=CFG["imgsz"], conf=0.25)
    print("Predictions done for", len(preds), "images")
else:
    print("No test images found")